## ▶ What you'll see when you run this
- Two RAG configs scored on the **RAG triad** (Context Relevance, Groundedness, Answer Relevance) and a printed **scorecard** showing which chunking wins.

**Time:** ~12 min · **Cost:** free (cheapest model: Gemini Flash / Claude Haiku) · **Keys:** none needed (heuristic fallback) — GEMINI_API_KEY or ANTHROPIC_API_KEY make the judge real

# Week 11 · Notebook 3 — Evaluating RAG: Measure, Don't Guess
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

You can *feel* that a RAG answer is good — but feelings don't scale and don't catch regressions. This notebook **measures** RAG quality with the **RAG triad**, then changes the chunking and **re-measures** to prove the change helped (or didn't).

We reuse the course's shared `eval_utils.py` (`llm_judge`, `scorecard`) so every week scores things the same way. Judging uses **Gemini or Claude**; with no key it falls back to a transparent heuristic so the notebook still runs.

## 0. Install

In [ ]:
!pip -q install langchain-text-splitters chromadb sentence-transformers \
    google-generativeai anthropic

## 1. Load API keys (optional)
With a key the judge is a real LLM; without one `llm_judge` degrades to a heuristic so nothing crashes.

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ('ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except Exception:
    pass
print('Anthropic key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('Gemini key set:', bool(os.environ.get('GEMINI_API_KEY')))

## 2. Import the shared eval helpers
`eval_utils.py` lives in the repo's top-level `tools/` folder. We search upward from the notebook for it so the import works whether you're local or in Colab. If it truly can't be found, we define a tiny inline fallback so the lesson still runs.

In [ ]:
import os, sys

def _find_tools_dir(start=None, max_up=6):
    here = os.path.abspath(start or os.getcwd())
    for _ in range(max_up):
        cand = os.path.join(here, 'tools', 'eval_utils.py')
        if os.path.exists(cand):
            return os.path.dirname(cand)
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

tools_dir = _find_tools_dir()
if tools_dir:
    sys.path.insert(0, tools_dir)
    from eval_utils import llm_judge, scorecard
    print('Imported eval_utils from:', tools_dir)
else:
    print('eval_utils.py not found nearby — using a minimal inline fallback.')
    import re, json
    def llm_judge(question, answer, rubric='', provider='gemini'):
        score = 3 if (answer and answer.strip()) else 1
        return {'score': score, 'reason': '(inline heuristic fallback)',
                'judge': 'fallback'}
    def scorecard(rows, score_key='score'):
        n = len(rows) or 1
        avg = sum(float(r.get(score_key, 0) or 0) for r in rows) / n
        print('SCORECARD', len(rows), 'cases  | avg', round(avg, 2))
        return {'n': len(rows), 'avg': avg}

## 3. A small knowledge base + gold questions
Same course handbook as Notebooks 1–2, plus a fixed set of **evaluation questions** (our test set). Real evals always pin a fixed question set so results are comparable run to run.

In [ ]:
COURSE_HANDBOOK = '''
Attendance policy: This is an online asynchronous course. Modules open Monday
and the weekly lab is due Sunday at 11:59 PM Pacific. You may miss deadlines
twice with no penalty using the two automatic 48-hour extensions.

AI-use policy: Students may use AI assistants such as Claude and Gemini on
assignments but must understand and disclose their use with an AI Use note.
Exams, including the midterm and final, are AI-restricted.

Grading: Labs and assignments are 50 percent, the midterm is 20 percent, and
the final project is 30 percent. The final project is due December 19.

Office hours: Tuesday 5 to 6 PM and Saturday 10 to 11 AM Pacific on Zoom.
Email the instructor with the subject prefix CSCI250 for a reply within 48 hours.
'''
EVAL_QUESTIONS = [
    'How is the final project weighted and when is it due?',
    'How many automatic extensions do I get and how long are they?',
    'What is the AI-use policy on exams?',
    'When are office hours?',
]
print(len(EVAL_QUESTIONS), 'eval questions ready')

## 4. A RAG function we can re-configure
Index the handbook with a given `chunk_size`/`overlap`, retrieve top-k, and answer. We keep generation simple and **offline-friendly**: the "answer" stitches the retrieved chunks together. (Swap in `answer_with_claude`/`answer_with_gemini` from Notebook 1 to evaluate a real generator the same way.)

In [ ]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

client = chromadb.Client()
_n = {'i': 0}

def build_rag(chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = [c.strip() for c in splitter.split_text(COURSE_HANDBOOK)]
    _n['i'] += 1
    col = client.create_collection(f"eval_{_n['i']}")
    col.add(documents=chunks,
            ids=[f'c{j}' for j in range(len(chunks))])

    def rag(question, k=3):
        res = col.query(query_texts=[question],
                        n_results=min(k, len(chunks)))
        retrieved = res['documents'][0]
        answer = ' '.join(retrieved)   # naive 'generator' for an offline demo
        return {'question': question, 'context': retrieved, 'answer': answer}
    return rag

## 5. The RAG triad — three judges, three questions
The RAG triad (popularized by TruLens) checks the **three links** in a RAG chain:

| Metric | Question it answers | Compares |
|---|---|---|
| **Context Relevance** | Did we retrieve the *right* chunks? | question ↔ context |
| **Groundedness** | Is the answer *supported* by the context (not made up)? | answer ↔ context |
| **Answer Relevance** | Does the answer actually address the question? | answer ↔ question |

If any link is weak you know *where* RAG broke — retrieval, faithfulness, or focus. We score each 1–5 with `llm_judge`.

In [ ]:
CONTEXT_RUBRIC = ('Rate 1-5 how RELEVANT the retrieved context is to the question. '
                  'Return ONLY JSON: {"score": <int 1-5>, "reason": "<short>"}.')
GROUND_RUBRIC  = ('Rate 1-5 how well the ANSWER is GROUNDED in (supported by) the '
                  'context — penalize anything not in the context. '
                  'Return ONLY JSON: {"score": <int 1-5>, "reason": "<short>"}.')
ANSWER_RUBRIC  = ('Rate 1-5 how RELEVANT the answer is to the question. '
                  'Return ONLY JSON: {"score": <int 1-5>, "reason": "<short>"}.')

def rag_triad(item):
    ctx = '\n'.join(item['context'])
    cr = llm_judge(item['question'], ctx, CONTEXT_RUBRIC)['score']
    gr = llm_judge('Context:\n' + ctx, item['answer'], GROUND_RUBRIC)['score']
    ar = llm_judge(item['question'], item['answer'], ANSWER_RUBRIC)['score']
    triad = (cr + gr + ar) / 3
    return {'context_relevance': cr, 'groundedness': gr,
            'answer_relevance': ar, 'score': round(triad, 2)}

## 6. Baseline run — measure config A
Evaluate small chunks (`chunk_size=120`). The `scorecard` prints a per-question report and an average triad score — our **baseline number to beat**.

In [ ]:
def evaluate(rag, label):
    rows = []
    for q in EVAL_QUESTIONS:
        item = rag(q, k=3)
        scores = rag_triad(item)
        rows.append({'q': q, **scores})
    print(f'\n### {label}')
    summary = scorecard(rows)
    return rows, summary

rag_a = build_rag(chunk_size=120, chunk_overlap=10)
rows_a, sum_a = evaluate(rag_a, 'Config A — small chunks (120/10)')

## 7. Change the chunking, then RE-MEASURE — config B
Larger chunks with more overlap (`chunk_size=300`, `overlap=60`) usually retrieve more complete context. We change **one thing** and re-run the **same** eval set so the comparison is fair.

In [ ]:
rag_b = build_rag(chunk_size=300, chunk_overlap=60)
rows_b, sum_b = evaluate(rag_b, 'Config B — larger chunks (300/60)')

## 8. Compare — did the change help?
Now we have **numbers**, not vibes. Whichever config has the higher average triad score wins; if they tie, the simpler/cheaper one wins.

In [ ]:
print('Config A avg triad:', round(sum_a['avg'], 2))
print('Config B avg triad:', round(sum_b['avg'], 2))
delta = sum_b['avg'] - sum_a['avg']
verdict = ('B is better' if delta > 0 else
           'A is better' if delta < 0 else 'tie')
print(f'Difference (B - A): {delta:+.2f}  ->  {verdict}')
print('\nNote: with NO API key both runs use the heuristic judge, so scores will')
print('look flat. Add GEMINI_API_KEY or ANTHROPIC_API_KEY to see the triad separate.')

## 9. Takeaways
- **Measure, don't guess.** A fixed question set + the **RAG triad** turns "feels good" into a number you can track.
- The triad **localizes** failures: low *Context Relevance* → fix retrieval/chunking; low *Groundedness* → tighten the grounding prompt; low *Answer Relevance* → fix the question framing or the generator.
- **Change one thing, re-measure.** That's how you improve a RAG system without fooling yourself.
- We use the **same** `eval_utils.llm_judge` / `scorecard` here as in Weeks 9, 13, 16, and 17 — evaluation is a course-long throughline, not a final-week topic.

**Your turn:** plug `answer_with_claude` from Notebook 1 into `build_rag`, re-run the triad on a real generator, and report which config you'd ship for your Final Project.